# LAB 07 - Transformer architecture

This notebook runs the required positional-encoding, multi-head, attention-diversity, sequence-length, and IMDb classification experiments. The implementation is in the neighbouring Python modules.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Lab7').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
LAB7_SRC = PROJECT_ROOT / 'Lab7' / 'src'
if not LAB7_SRC.exists():
    raise FileNotFoundError('Không tìm thấy Lab7/src. Hãy mở notebook từ repository NLP.')
sys.path.insert(0, str(LAB7_SRC))
from run_lab7 import (run_positional_experiment, run_attention_experiments,
                      run_classification, find_dataset, seed_everything)
from multi_head_self_attention import MultiHeadSelfAttention
from positional_encoding import PositionalEncoding
from transformer_classifier import TransformerClassifier

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Positional encoding and permutation experiment

In [ ]:
print(run_positional_experiment())
pe = PositionalEncoding(d_model=4, max_len=3)
print('PE positions 0, 1, 2:')
print(pe.pe[0])
print('The distance should be close to zero without PE and non-zero with PE.')

## 2. Number and diversity of attention heads

With fixed `d_model=128`, increasing the number of heads does not increase the output representation dimension; it changes `d_head = 128 / n_heads`.

In [ ]:
for row in run_attention_experiments():
    print(row)

torch.manual_seed(42)
attention = MultiHeadSelfAttention(d_model=32, n_heads=4).eval()
x = torch.randn(1, 12, 32)
with torch.no_grad():
    _, weights = attention(x)
head_similarity = torch.corrcoef(weights[0].reshape(4, -1))
print('Attention shape [batch, heads, T, T]:', tuple(weights.shape))
print('Head correlation matrix:\n', head_similarity)
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for index, axis in enumerate(axes):
        axis.imshow(weights[0, index].numpy(), aspect='auto', cmap='viridis')
        axis.set_title(f'Head {index + 1}')
    fig.tight_layout()
    fig.savefig(PROJECT_ROOT / 'Lab7' / 'attention_heads.png', dpi=150)
    plt.show()
except ImportError:
    print('matplotlib chưa được cài; bỏ qua biểu đồ attention.')

## 3. Sequence-length cost

Self-attention stores a `T x T` matrix per head. The number of attention entries is therefore quadratic in sequence length.

In [ ]:
for length in (32, 64, 128, 256):
    print({'max_length': length, 'attention_entries_per_head': length * length,
           'relative_to_32': (length / 32) ** 2})

## 4. IMDb classification: LSTM versus Transformer

The runner uses the same split, vocabulary, optimizer, batch size, and sequence length for both models. Set `LAB7_MAX_SAMPLES`, `LAB7_EPOCHS`, `LAB7_MAX_LENGTH`, or `LAB7_BATCH_SIZE` before running if a shorter experiment is desired.

In [ ]:
print('Dataset:', find_dataset())
rows, history = run_classification()
for row in rows:
    print(row)
try:
    import matplotlib.pyplot as plt
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['valid_loss'], label='validation')
    plt.xlabel('Epoch')
    plt.ylabel('Cross-entropy loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'Lab7' / 'learning_curves.png', dpi=150)
    plt.show()
except ImportError:
    pass

## 5. Shape checks

For `B=8`, `T=100`, `d_model=128`, and `n_heads=4`: Q, K, and V have shape `[8, 4, 100, 32]`; attention scores have shape `[8, 4, 100, 100]`. The implementation checks the mask shape and preserves these dimensions.